# Notebook 1 — Preprocesado

Este notebook aplica el pipeline de limpieza y preprocesado al dataset crudo
`rendimiento_estudiantes.csv` y genera los ficheros de entrenamiento y test para
las tareas de **clasificación** y **regresión**.

Toda la lógica de limpieza está implementada en `src/clean_data.py`; aquí nos
limitamos a orquestar las llamadas y justificar las decisiones principales.

## Índice

1. Carga del dataset crudo.
2. Exploración rápida inicial.
3. Limpieza paso a paso y justificación.
4. Generación de los datasets para clasificación y regresión.
5. Verificación final.

## 0. Configuración del entorno

Añadimos `src/` al `sys.path` para poder importar los módulos del proyecto
como si fueran una librería.

In [1]:
import sys
from pathlib import Path

# Raíz del proyecto (padre de notebooks/)
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
import numpy as np

import clean_data
from configuracion import RAW_DATA_PATH, PROCESSED_DIR

print(f'Raíz del proyecto: {ROOT}')
print(f'Dataset crudo:     {RAW_DATA_PATH}')
print(f'Directorio salida: {PROCESSED_DIR}')

Raíz del proyecto: c:\Users\mb6ja\Downloads\proyecto
Dataset crudo:     C:\Users\mb6ja\Downloads\proyecto\data\rendimiento_estudiantes.csv
Directorio salida: C:\Users\mb6ja\Downloads\proyecto\data\processed


## 1. Carga del dataset crudo

El fichero original usa `;` como separador. Lo cargamos tal cual viene, sin
modificaciones todavía.

In [2]:
df = pd.read_csv(RAW_DATA_PATH, sep=';')
print(f'Shape: {df.shape[0]} filas × {df.shape[1]} columnas')
df.head()

Shape: 4424 filas × 37 columnas


,estado_civil,modo_solicitud,orden_solicitud,curso,asistencia_diurna_vespertina,cualificacion_previa,nota_cualificacion_previa,nacionalidad,cualificacion_madre,cualificacion_padre,...,asignaturas_2sem_convalidadas,asignaturas_2sem_matriculadas,asignaturas_2sem_evaluadas,asignaturas_2sem_aprobadas,nota_media_2sem,asignaturas_2sem_sin_evaluacion,tasa_desempleo,tasa_inflacion,pib,objetivo
0,soltero/a,2a_fase_contingente_general,5,animacion_y_diseno_multimedia,diurna,educacion_secundaria,122.0,portuguesa,educacion_basica_3er_ciclo_o_equivalente,otro_11o_ano,...,0,0,0,0,0.000000,0,10.8,1.4,1.74,abandono
1,soltero/a,estudiante_internacional_licenciatura,1,turismo,diurna,educacion_secundaria,160.0,portuguesa,educacion_secundaria_12o_ano_o_equivalente,educacion_superior_grado,...,0,6,6,6,13.666667,0,13.9,-0.3,0.79,graduado
2,soltero/a,1a_fase_contingente_general,5,diseno_de_comunicacion,diurna,educacion_secundaria,122.0,portuguesa,educacion_basica_1er_ciclo_o_equivalente,educacion_basica_1er_ciclo_o_equivalente,...,0,6,0,0,0.000000,0,10.8,1.4,1.74,abandono
3,soltero/a,2a_fase_contingente_general,2,periodismo_y_comunicacion,diurna,educacion_secundaria,122.0,portuguesa,educacion_basica_2o_ciclo_o_equivalente,educacion_basica_1er_ciclo_o_equivalente,...,0,6,10,5,12.400000,0,9.4,-0.8,-3.12,graduado
4,casado/a,mayor_de_23_anos,1,servicio_social_nocturno,vespertina,educacion_secundaria,100.0,portuguesa,educacion_basica_1er_ciclo_o_equivalente,educacion_basica_2o_ciclo_o_equivalente,...,0,6,6,6,13.000000,0,13.9,-0.3,0.79,graduado


## 2. Exploración rápida

Antes de limpiar, comprobamos tipos, cardinalidades y la distribución de la
variable objetivo.

In [3]:
# Tipos de variables
n_num = df.select_dtypes(include=['int64', 'float64']).shape[1]
n_cat = df.select_dtypes(include=['object']).shape[1]
print(f'Variables numéricas:   {n_num}')
print(f'Variables categóricas: {n_cat}')
print(f'Valores nulos totales: {df.isna().sum().sum()}')

print('\nDistribución de la variable objetivo:')
print(df['objetivo'].value_counts())
print('\nProporción:')
print((df['objetivo'].value_counts(normalize=True) * 100).round(2))

Variables numéricas:   19
Variables categóricas: 18
Valores nulos totales: 0

Distribución de la variable objetivo:
objetivo
graduado       2209
abandono       1421
matriculado     794
Name: count, dtype: int64

Proporción:
objetivo
graduado       49.93
abandono       32.12
matriculado    17.95
Name: proportion, dtype: float64


La variable objetivo está **desbalanceada**: `graduado` es ~50 %, `abandono`
~32 % y `matriculado` sólo ~18 %. Esto condicionará la elección de métricas
en clasificación (usaremos **F1 macro**, no accuracy).

Aunque `isna().sum()` devuelve 0, sabemos que hay nulos *codificados como
texto* (`desconocido`, `en_blanco`, `otra_situacion`) en varias variables
categóricas — ese es el primer paso de limpieza.

## 3. Limpieza paso a paso

Aplicamos cada transformación por separado para ver su efecto, y veremos que `limpiar_dataset` incluye todo.

### 3.1 Unificación de valores nulos 

Tres etiquetas (`desconocido`, `en_blanco`, `otra_situacion`) expresan la
misma idea: *información no disponible*. Las unificamos, teniendo en cuenta
que la falta del dato es en sí misma informativa a la hora de clasificar.

In [4]:
df_tmp = clean_data.unificar_nulls(df)
etiqueta = 'informacion_no_disponible'
n_nuevos = (df_tmp == etiqueta).sum().sum()
print(f'Celdas marcadas como "{etiqueta}": {n_nuevos}')

Celdas marcadas como "informacion_no_disponible": 413


### 3.2 Codificación de la variable objetivo

Mapeamos `abandono→0`, `matriculado→1`, `graduado→2`. Es necesario para
evitar errores de casting en KNN (que requiere targets numéricos).

In [5]:
df_tmp = clean_data.codificar_objetivo(df_tmp)
print(df_tmp['objetivo'].value_counts().sort_index())

objetivo
0    1421
1     794
2    2209
Name: count, dtype: int64


### 3.3 Corrección de inconsistencias académicas

Detectamos tres tipos de inconsistencias en las variables de asignaturas:

- Filas con más de **20 asignaturas evaluadas** en un semestre: claramente
  errores de registro, se eliminan.
- **Evaluadas > matriculadas** y **aprobadas > evaluadas**: lógicamente
  imposibles; se recortan con `np.minimum`.
- **Convalidadas > 8**: valores extremos poco realistas; se acotan a 8.

In [6]:
n_antes = len(df_tmp)
df_tmp = clean_data.limpiar_vals_academicas(df_tmp)
print(f'Filas eliminadas por outliers en evaluadas: {n_antes - len(df_tmp)}')
print(f'Shape actual: {df_tmp.shape}')

Filas eliminadas por outliers en evaluadas: 75
Shape actual: (4349, 37)


### 3.4 Reducción de cardinalidad categórica

Varias columnas categóricas tienen muchas etiquetas con frecuencia muy baja,
lo que produciría demasiadas dimensiones tras el one-hot encoding y
un alto riesgo de overfitting a categorías raras. Aplicamos tres estrategias:

1. **`cualificacion_previa`**: se englobal en 5 grupos de alto nivel
   (`superior`, `tecnico`, `basica`, `secundaria`, `otros_incompletos`).
2. **`modo_solicitud`**: se agrupan en 5 vías de acceso principales.
3. **Información de padres** (`ocupacion_*`, `cualificacion_*`): categorías
   con menos de 20 instancias se unen bajo la etiqueta `otras_categorias`.

In [7]:
print('Cardinalidad ANTES de reducir:')
print(f"  cualificacion_previa: {df_tmp['cualificacion_previa'].nunique()}")
print(f"  modo_solicitud:       {df_tmp['modo_solicitud'].nunique()}")
print(f"  ocupacion_madre:      {df_tmp['ocupacion_madre'].nunique()}")
print(f"  ocupacion_padre:      {df_tmp['ocupacion_padre'].nunique()}")

df_tmp = clean_data.limpiar_cualificacion_previa(df_tmp)
df_tmp = clean_data.limpiar_modo_solicitud(df_tmp)
df_tmp = clean_data.limpiar_info_padres(df_tmp, min_frecuencia=20)

print('\nCardinalidad DESPUÉS de reducir:')
print(f"  cualificacion_previa: {df_tmp['cualificacion_previa'].nunique()}")
print(f"  modo_solicitud:       {df_tmp['modo_solicitud'].nunique()}")
print(f"  ocupacion_madre:      {df_tmp['ocupacion_madre'].nunique()}")
print(f"  ocupacion_padre:      {df_tmp['ocupacion_padre'].nunique()}")

Cardinalidad ANTES de reducir:
  cualificacion_previa: 17
  modo_solicitud:       18
  ocupacion_madre:      31
  ocupacion_padre:      45

Cardinalidad DESPUÉS de reducir:
  cualificacion_previa: 5
  modo_solicitud:       5
  ocupacion_madre:      13
  ocupacion_padre:      13


### 3.5 Eliminación de `nacionalidad`

La variable `nacionalidad` es altamente desbalanceada (la gran mayoría es
portuguesa) y redundante con la variable binaria `internacional`. La
eliminamos para simplificar.

In [8]:
df_tmp = df_tmp.drop(columns=['nacionalidad'])
print(f'Shape tras eliminar nacionalidad: {df_tmp.shape}')

Shape tras eliminar nacionalidad: (4349, 36)


### 3.6 Equivalencia con `limpiar_dataset`

Todas las llamadas anteriores están encapsuladas en `clean_data.limpiar_dataset`.
Verificamos que produce exactamente el mismo resultado partiendo del dataset
crudo.

In [9]:
df_limpio = clean_data.limpiar_dataset(df)
assert df_limpio.shape == df_tmp.shape, 'Shapes distintos!'
assert list(df_limpio.columns) == list(df_tmp.columns), 'Columnas distintas!'
print(f'OK: limpiar_dataset produce el mismo resultado ({df_limpio.shape}).')

OK: limpiar_dataset produce el mismo resultado ((4349, 36)).


## 4. Generación de los datasets de salida

A partir del DataFrame limpio generamos dos datasets distintos, uno por tarea.

### 4.1 Dataset de clasificación

Se mantienen **todas las variables**, incluyendo las notas 0.0, porque son un
indicador directo de abandono temprano y por tanto una señal clave para
predecir la variable objetivo.

Split **80/20 estratificado** por `objetivo` para preservar las proporciones
de clase en train y test.

In [10]:
df_clf_train, df_clf_test = clean_data.generar_data_clasificacion(
    df_limpio,
    output_dir=str(PROCESSED_DIR),
    test_size=0.20,
    random_state=0,
)
print(f'Train clasificación: {df_clf_train.shape[0]} filas')
print(f'Test  clasificación: {df_clf_test.shape[0]} filas')
print(f'\nProporción de clases en train:\n{(df_clf_train["objetivo"].value_counts(normalize=True)*100).round(2)}')
print(f'\nProporción de clases en test:\n{(df_clf_test["objetivo"].value_counts(normalize=True)*100).round(2)}')

Train clasificación: 3479 filas
Test  clasificación: 870 filas

Proporción de clases en train:
objetivo
2    49.70
0    32.22
1    18.08
Name: proportion, dtype: float64

Proporción de clases en test:
objetivo
2    49.77
0    32.18
1    18.05
Name: proportion, dtype: float64


### 4.2 Dataset de regresión

La variable objetivo es **`nota_media_2sem`**. Dos cuidados metodológicos:

**Anti data-leakage.** Eliminamos todas las variables del 2º semestre distintas
del target (`asignaturas_2sem_*`), y también `objetivo` (estado final del
alumno, demasiado reciente para el target).

**Anti sesgo por abandono.** Excluimos los alumnos con nota 0.0 en el 1er o
el 2º semestre. Estos no son estudiantes con notas bajas, sino abandonos durante el año:
su nota de 0.0 no refleja rendimiento, sino directamente ausencia de evaluacion.
Mantenerlos haría que el modelo tuviera que ajustar una función
continua sobre dos poblaciones distintas (una de ellas totalmente distribuida en el 0.0) y sesgaría las
predicciones hacia abajo.

In [11]:
df_reg_train, df_reg_test = clean_data.generar_data_regresion(
    df_limpio,
    output_dir=str(PROCESSED_DIR),
    test_size=0.20,
    random_state=42,
)
print(f'Train regresión: {df_reg_train.shape[0]} filas, {df_reg_train.shape[1]} columnas')
print(f'Test  regresión: {df_reg_test.shape[0]} filas')
print(f'\nTarget nota_media_2sem — rango: {df_reg_train["nota_media_2sem"].min():.2f} a {df_reg_train["nota_media_2sem"].max():.2f}')
print(f'Target nota_media_2sem — media: {df_reg_train["nota_media_2sem"].mean():.2f}, std: {df_reg_train["nota_media_2sem"].std():.2f}')

Train regresión: 2750 filas, 30 columnas
Test  regresión: 688 filas

Target nota_media_2sem — rango: 10.00 a 18.57
Target nota_media_2sem — media: 12.75, std: 1.37


## 5. Verificación final

Comprobamos que los ficheros se han escrito correctamente en
`data/processed/`.

In [13]:
for f in sorted(PROCESSED_DIR.glob('*.csv')):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:50s}  {size_kb:>8.1f} KB')

# Verificación anti-leakage en el dataset de regresión
cols_reg = pd.read_csv(PROCESSED_DIR / 'datos_limpios_regresion_train.csv', sep=';').columns
leak = [c for c in cols_reg if '2sem' in c.lower() and c != 'nota_media_2sem']
assert len(leak) == 0, f'LEAKAGE: {leak}'
print('\nOK: ninguna variable del 2º semestre en el dataset de regresión.')

  datos_limpios_clasificacion_test.csv                   274.8 KB
  datos_limpios_clasificacion_train.csv                 1095.4 KB
  datos_limpios_regresion_test.csv                       211.5 KB
  datos_limpios_regresion_train.csv                      842.4 KB

OK: ninguna variable del 2º semestre en el dataset de regresión.


---

Con esto los datos están listos. Los siguientes notebooks asumen que estos
cuatro ficheros existen en `data/processed/`:

- `datos_limpios_clasificacion_train.csv`
- `datos_limpios_clasificacion_test.csv`
- `datos_limpios_regresion_train.csv`
- `datos_limpios_regresion_test.csv`